# 04 — Aggregate monthly NDVI to annual rasters

Create the annual NDVI series used by trajectory, state, and performance. Notebooks 02 and 03 are not prerequisites for this branch.


## Inputs, outputs, and execution order

**Inputs:** Monthly rolling NDVI rasters from 01 on an identical grid, with 12 files per calendar year.

**Outputs:** `annual_ndvi/ndvi_annual_median_YYYY.tif` by default, used by 05, 06, and 07.

Run cells from top to bottom in a fresh kernel. Open this notebook beside `workflow_config.json` and `workflow_support.py`. The configuration uses Mion study settings; the original boundary and final criteria table were not supplied. See the README for the small example and for adapting the workflow.

### How to interpret this step

The default annual metric is the **median of monthly rolling NDVI**. It is different from annual peak NDVI, phenological vPOS, an annual integral, or net primary production. A year with a missing monthly file is skipped. `require_all_months=False` permits a pixel to have fewer than 12 valid monthly values even though all 12 files must exist.


## 1. Import packages

Use the Digital Earth Africa Sandbox for steps that access the data cube. Local raster processing uses the packages listed in `environment.yml`.


In [ ]:
import os
import re
import gc
import glob
import numpy as np
import rasterio
from rasterio.windows import Window


## 2. Load shared settings

The settings below describe the run. Edit the JSON configuration before continuing; later parameter cells explain the settings used by this notebook.


In [ ]:
# Edit workflow_config.json once, then restart and rerun each notebook.
from workflow_support import load_config, require_files, load_aoi, assert_same_grid
cfg = load_config()
print(f"Study: {cfg['study_id']} | Years: {cfg['start_year']}-{cfg['end_year']}")
print(f"Output directory: {cfg['output_dir']}")


## 3. Define aggregation helpers

Rasters are processed in spatial windows to reduce RAM use. Duplicate months and mismatched grids are rejected. GeoTIFF tile dimensions must be multiples of 16, including for small research areas.


In [ ]:
def yyyymm_from_path(path: str):
    """
    Extract YYYYMM from filename.
    Expects any 6-digit YYYYMM pattern in the basename.
    """
    base = os.path.basename(path)
    m = re.search(r"(19|20)\d{2}(0[1-9]|1[0-2])", base)
    if not m:
        raise ValueError(f"Could not parse YYYYMM from: {base}")
    yyyymm = m.group(0)
    return int(yyyymm[:4]), int(yyyymm[4:6])


def group_monthlies_by_year(monthly_paths):
    """
    Group monthly raster paths as:
    {year: {month: path}}
    """
    groups = {}
    for p in monthly_paths:
        y, m = yyyymm_from_path(p)
        if m in groups.setdefault(y, {}):
            raise ValueError(f"Duplicate monthly input: {y}-{m:02d}")
        groups[y][m] = p
    return groups


def _safe_aggregate(M, annual_metric):
    """
    Aggregate a monthly stack M of shape (12, h, w) with NaN support.
    Avoids noisy warnings on all-NaN slices.
    """
    valid_any = np.isfinite(M).any(axis=0)

    A = np.full(M.shape[1:], np.nan, dtype="float32")

    if not valid_any.any():
        return A

    if annual_metric == "mean":
        A[valid_any] = np.nanmean(M[:, valid_any], axis=0)
    elif annual_metric == "median":
        A[valid_any] = np.nanmedian(M[:, valid_any], axis=0)
    elif annual_metric == "max":
        A[valid_any] = np.nanmax(M[:, valid_any], axis=0)
    elif annual_metric == "sum":
        A[valid_any] = np.nansum(M[:, valid_any], axis=0)
    else:
        raise ValueError("annual_metric must be mean|median|max|sum")

    return A.astype("float32")


def aggregate_annual_from_monthly(
    monthly_paths,
    out_dir,
    annual_metric="median",        # "mean" | "median" | "max" | "sum"
    require_all_months=False,      # True => pixel must be valid in all 12 months
    blocksize=512,
    skip_existing=True,
):
    """
    Create annual GeoTIFFs from monthly GeoTIFFs, one per year.

    Parameters
    ----------
    monthly_paths : list[str]
        List of monthly GeoTIFF paths.
    out_dir : str
        Output folder for annual rasters.
    annual_metric : str
        One of: mean, median, max, sum.
    require_all_months : bool
        If True, only pixels with all 12 months valid are aggregated.
    blocksize : int
        Processing block size.
    skip_existing : bool
        If True, skip output files that already exist.

    Returns
    -------
    list[str]
        List of written annual GeoTIFF paths.
    """
    assert_same_grid(monthly_paths)
    os.makedirs(out_dir, exist_ok=True)

    groups = group_monthlies_by_year(monthly_paths)
    years = sorted(groups.keys())

    if not years:
        raise ValueError("No monthly rasters found.")

    # Use first available file as template
    first_path = None
    for y in years:
        if groups[y]:
            first_path = next(iter(groups[y].values()))
            break

    if first_path is None:
        raise ValueError("No valid monthly rasters found.")

    with rasterio.open(first_path) as src0:
        profile = src0.profile.copy()
        H, W = src0.height, src0.width

    float_profile = profile.copy()
    float_profile.update(
        dtype="float32",
        count=1,
        nodata=np.nan,
        compress="deflate",
        tiled=True,
        blockxsize=blocksize,
        blockysize=blocksize,
    )

    annual_paths = []

    for y in years:
        month_map = groups[y]
        missing = [m for m in range(1, 13) if m not in month_map]
        if missing:
            print(f"Skipping year {y}: missing months {missing}")
            continue

        out_path = os.path.join(out_dir, f"ndvi_annual_{annual_metric}_{y}.tif")

        if skip_existing and os.path.exists(out_path):
            print(f"Skipping existing file: {out_path}")
            annual_paths.append(out_path)
            continue

        print(f"Processing year {y} -> {out_path}")

        # Open all 12 monthly rasters once for the year
        srcs = []
        try:
            for m in range(1, 13):
                srcs.append(rasterio.open(month_map[m]))

            with rasterio.open(out_path, "w", **float_profile) as dst:
                dst.update_tags(Year=str(y), AnnualMetric=annual_metric)

                for row0 in range(0, H, blocksize):
                    for col0 in range(0, W, blocksize):
                        h = min(blocksize, H - row0)
                        w = min(blocksize, W - col0)
                        win = Window(col0, row0, w, h)

                        stack = []
                        for src in srcs:
                            a = src.read(1, window=win).astype("float32", copy=False)

                            if src.nodata is not None:
                                a = np.where(a == src.nodata, np.nan, a)

                            stack.append(a)

                        M = np.stack(stack, axis=0)  # shape: (12, h, w)

                        if require_all_months:
                            ok = np.isfinite(M).all(axis=0)
                            M = np.where(ok[None, :, :], M, np.nan)

                        A = _safe_aggregate(M, annual_metric)
                        dst.write(A, 1, window=win)

                        del stack, M, A

            annual_paths.append(out_path)
            print(f"Wrote annual raster: {out_path}")

        finally:
            for src in srcs:
                try:
                    src.close()
                except Exception:
                    pass

            gc.collect()

    return annual_paths


## 4. Run annual aggregation

File selection is limited to the configured years and version. Check that the number of annual outputs equals the number of requested years before starting the indicators.


In [ ]:
trends_output = f"{cfg['output_dir']}/data"
monthly_paths = [p for year in range(cfg["start_year"], cfg["end_year"] + 1)
                 for p in sorted(glob.glob(f"{trends_output}/ndvi_roll3_{year}??_{cfg['file_version']}.tif"))]
annual_files = aggregate_annual_from_monthly(
    monthly_paths=monthly_paths,
    out_dir=f"{cfg['output_dir']}/annual_ndvi",
    annual_metric=cfg["annual_metric"],
    require_all_months=cfg["require_all_months"],
    blocksize=cfg["block_size"],
    skip_existing=cfg["skip_existing"],
)
expected = [f"{cfg['output_dir']}/annual_ndvi/ndvi_annual_{cfg['annual_metric']}_{year}.tif"
            for year in range(cfg["start_year"], cfg["end_year"] + 1)]
require_files(*expected)
print(f"Annual files available: {len(expected)}")


## Acknowledgements

We gratefully acknowledge the [Group on Earth Observations Land Degradation Neutrality (GEO-LDN) initiative](https://geo-ldn.org/) for its support of this research and the development of this land productivity monitoring workflow.

We acknowledge **Digital Earth Africa (DE Africa)** for providing the open satellite data, analytical tools, and Sandbox platform used in this work.

We also acknowledge the [University of Energy and Natural Resources (UENR), Ghana](https://uenr.edu.gh/), for its support of this research.


## Check the outputs before continuing

- Review warnings and errors, and confirm that every requested year or reporting period was written.
- Open an output raster and inspect its CRS, extent, nodata mask, and value range.
- Keep the configuration and software versions with your results. Run `python scripts/capture_environment.py` from the repository root.
- If you change the AOI, dates, metric, or thresholds, use a fresh output directory to avoid reusing older results.

See [README.md](README.md), [data requirements](docs/DATA_REQUIREMENTS.md), and [method details](docs/METHODS.md).

The supplied notebooks identify their code as Apache License 2.0; retain the original Digital Earth Africa attribution when reusing it. Dataset licences and acknowledgements apply separately. Use the Issues tab of the repository hosting this workflow for workflow-specific questions.

**Validation status:** documentation, syntax, and synthetic-data checks are described in `docs/VALIDATION.md`. This revised workflow has not been executed against the live Digital Earth Africa data cube. A displayed current date or package version is not evidence of a successful full run.
